In [1]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceM4/FineVisionMax", split="train", streaming=True)
print(next(iter(dataset)))

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/10000 [00:00<?, ?it/s]

{'images': [<PIL.PngImagePlugin.PngImageFile image mode=RGB size=1506x1102 at 0xEC954E9D5600>], 'texts': [{'user': 'What is the primary key of the Equipment entity?', 'assistant': 'The primary key of the Equipment entity is EquipmentID, which uniquely identifies each piece of equipment in the diagram.\nAnswer: EquipmentID'}, {'user': 'What is the relationship between Equipment and Maintenance Record?', 'assistant': 'The relationship between Equipment and Maintenance Record is labeled as "Maintained By", indicating that one piece of equipment can have multiple maintenance records associated with it.\nAnswer: Maintained By'}, {'user': 'How many relationships does the Supplier entity have?', 'assistant': 'The Supplier entity has two relationships: "Supplied By" with Equipment and "Delivered By" with Logistics Company, indicating it serves as a midpoint in both flows.\nAnswer: 2'}, {'user': 'What is the cardinality of the relationship between Equipment and Supplier?', 'assistant': 'The car

In [30]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoProcessor, AutoTokenizer, AutoModelForCausalLM

class VisionLanguageModel(nn.Module):
    def __init__(self, vision_encoder_ckpt, language_model_ckpt, modality_input_dim=768, modality_output_dim=576):
        super().__init__()
        self.vision_encoder = AutoModel.from_pretrained(vision_encoder_ckpt).vision_model  # only take vision backbone
        self.vision_processor = AutoProcessor.from_pretrained(vision_encoder_ckpt)
        self.modality_projector = nn.Linear(modality_input_dim, modality_output_dim, bias=False)
        self.tokenizer = AutoTokenizer.from_pretrained(language_model_ckpt)
        self.llm = AutoModelForCausalLM.from_pretrained(language_model_ckpt)

    def forward(self, text, image, labels=None):
        processed_img = self.vision_processor(images=[image], return_tensors="pt").to(self.llm.device)
        image_embd = self.vision_encoder(**processed_img).last_hidden_state
        image_embd = self.modality_projector(image_embd).to(dtype=self.llm.dtype)  # match LLM dtype

        input_ids = self.tokenizer(text, return_tensors="pt").input_ids.to(self.llm.device)
        token_embd = self.llm.model.embed_tokens(input_ids)

        combined_embd = torch.cat((image_embd, token_embd), dim=1) # Concatenate image embeddings to token embeddings

        logits = self.llm(inputs_embeds=combined_embd).logits

        shift_logits = logits[:, image_embd.size(1):-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()

        loss = nn.functional.cross_entropy(shift_logits.reshape(-1, shift_logits.size(-1)), shift_labels.reshape(-1))

        return logits, loss

In [31]:
vision_encoder_ckpt = "google/siglip2-base-patch16-256"
language_model_ckpt = "HuggingFaceTB/SmolLM2-135M-Instruct"

vlm = VisionLanguageModel(vision_encoder_ckpt, language_model_ckpt).to("cuda")

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [28]:
from transformers.image_utils import load_image

image = load_image("https://huggingface.co/datasets/merve/coco/resolve/main/val2017/000000000285.jpg")
text = "What does this say?"
vlm(text, image)

torch.Size([1, 256, 576])


(tensor([[[15.0000,  6.1875,  9.3750,  ...,  6.9375, 10.7500,  2.0469],
          [22.8750, 11.6875, 14.8750,  ..., 17.8750, 19.1250, 16.8750],
          [20.7500,  5.8125,  7.5312,  ..., 12.5000, 15.8750, 10.5000],
          ...,
          [13.9375,  2.1250,  3.8906,  ..., 10.4375, 12.1250,  8.1250],
          [13.6250,  3.0625,  6.1875,  ...,  8.0625, 12.3750,  5.6562],
          [14.3125,  5.4062, 10.1250,  ...,  9.1250, 12.5625,  6.8750]]],
        device='cuda:0', dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>),
 tensor(4.8125, device='cuda:0', dtype=torch.bfloat16,
        grad_fn=<NllLossBackward0>))

In [10]:
sample = next(iter(dataset))

def apply_chat_template_to_sample(sample):
  messages_list = []
  for data_dict in sample['texts']:
      messages_list.append({'role': 'user', 'content': data_dict['user']})
      messages_list.append({'role': 'assistant', 'content': data_dict['assistant']})
  text = vlm.tokenizer.apply_chat_template(messages_list, tokenize=False)
  return text

text = apply_chat_template_to_sample(sample)


In [11]:
sample

{'images': [<PIL.PngImagePlugin.PngImageFile image mode=RGB size=1506x1102>],
 'texts': [{'user': 'What is the primary key of the Equipment entity?',
   'assistant': 'The primary key of the Equipment entity is EquipmentID, which uniquely identifies each piece of equipment in the diagram.\nAnswer: EquipmentID'},
  {'user': 'What is the relationship between Equipment and Maintenance Record?',
   'assistant': 'The relationship between Equipment and Maintenance Record is labeled as "Maintained By", indicating that one piece of equipment can have multiple maintenance records associated with it.\nAnswer: Maintained By'},
  {'user': 'How many relationships does the Supplier entity have?',
   'assistant': 'The Supplier entity has two relationships: "Supplied By" with Equipment and "Delivered By" with Logistics Company, indicating it serves as a midpoint in both flows.\nAnswer: 2'},
  {'user': 'What is the cardinality of the relationship between Equipment and Supplier?',
   'assistant': 'The ca

In [12]:
text

'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWhat is the primary key of the Equipment entity?<|im_end|>\n<|im_start|>assistant\nThe primary key of the Equipment entity is EquipmentID, which uniquely identifies each piece of equipment in the diagram.\nAnswer: EquipmentID<|im_end|>\n<|im_start|>user\nWhat is the relationship between Equipment and Maintenance Record?<|im_end|>\n<|im_start|>assistant\nThe relationship between Equipment and Maintenance Record is labeled as "Maintained By", indicating that one piece of equipment can have multiple maintenance records associated with it.\nAnswer: Maintained By<|im_end|>\n<|im_start|>user\nHow many relationships does the Supplier entity have?<|im_end|>\n<|im_start|>assistant\nThe Supplier entity has two relationships: "Supplied By" with Equipment and "Delivered By" with Logistics Company, indicating it serves as a midpoint in both flows.\nAnswer: 2<|im_end|>\n<|im_start|>

In [13]:
vlm(text, sample['images'])

torch.Size([1, 256, 576])


(tensor([[[18.7500, 11.0000, 15.1250,  ..., 12.6875, 15.3750,  8.4375],
          [19.5000, 10.0625, 14.5625,  ..., 17.5000, 16.5000, 12.5625],
          [16.7500,  9.6875, 13.6250,  ..., 11.4375, 14.0000, 10.3750],
          ...,
          [17.8750, 25.5000, 32.7500,  ..., 16.0000, 11.5000,  9.9375],
          [24.6250, 27.6250, 29.8750,  ..., 13.6875, 13.0000, 11.3750],
          [13.5000, 31.8750, 31.6250,  ..., 10.7500,  8.9375,  8.5625]]],
        device='cuda:0', dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>),
 tensor(1.9062, device='cuda:0', dtype=torch.bfloat16,
        grad_fn=<NllLossBackward0>))

In [32]:
import json
import torch.optim as optim
optimizer = optim.AdamW(vlm.parameters(), lr=1e-5)

step = 0
losses = []
for sample in dataset:
    text = apply_chat_template_to_sample(sample)
    if len(text) > 3000 or len(sample['images']) != 1:
      continue
    images = [sample['images'][0].convert('RGB')]
    step += 1

    optimizer.zero_grad()
    logits, loss = vlm(text, images)
    losses.append(loss)

    loss.backward()
    optimizer.step()

    if step%100 == 0:
        print(f"step {step} | loss {loss.item():.4f}")

    if step > 5000:
      json.dump([round(l.item(), 4) for l in losses], open("chapter_3_loss.json", "w"))
      break

step 100 | loss 1.9766
step 200 | loss 2.7969
step 300 | loss 2.0938
step 400 | loss 5.1875
step 500 | loss 2.5625
step 600 | loss 2.5000
step 700 | loss 1.3516
step 800 | loss 1.0000
step 900 | loss 1.5625
step 1000 | loss 1.7812
step 1100 | loss 2.4688
step 1200 | loss 1.6250
step 1300 | loss 2.1094
step 1400 | loss 1.1641
step 1500 | loss 0.9219
step 1600 | loss 1.5000
step 1700 | loss 2.2969
step 1800 | loss 1.4219
step 1900 | loss 2.5781
step 2000 | loss 1.8828
step 2100 | loss 0.6445
step 2200 | loss 0.7930
step 2300 | loss 1.5078
step 2400 | loss 2.2969
step 2500 | loss 2.2812
step 2600 | loss 1.7031
step 2700 | loss 1.9531
step 2800 | loss 2.4375
step 2900 | loss 0.7578
step 3000 | loss 3.6562
step 3100 | loss 4.1250
step 3200 | loss 1.8281
step 3300 | loss 2.3125
step 3400 | loss 1.6406
step 3500 | loss 2.4375
step 3600 | loss 1.8047
step 3700 | loss 1.9531
step 3800 | loss 2.0000
step 3900 | loss 1.7812
step 4000 | loss 1.2891
step 4100 | loss 1.8516
step 4200 | loss 1.8438
s

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("HuggingFaceM4/FineVisionMax", split="train", streaming=True)
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

def tokenize_sample(sample):
    messages_list = []
    for data_dict in sample['texts']:
        messages_list.append({'role': 'user', 'content': data_dict['user']})
        messages_list.append({'role': 'assistant', 'content': data_dict['assistant']})
    tokens = tokenizer.apply_chat_template(messages_list, tokenize=True)
    return tokens['input_ids']

max_length = 2048
curr_sample = []
batch = []
max_batch_size = 8
for sample in dataset:
    tokens = tokenize_sample(sample)
    if len(tokens) > max_length:
        continue  # Skip samples that are too long
    if len(curr_sample) + len(tokens) < max_length:
        curr_sample.extend(tokens)
    else:
        if len(batch) == max_batch_size:
            break # We've reached the max batch size, so we need to break
        batch.append(curr_sample)
        curr_sample = tokens

Resolving data files:   0%|          | 0/10000 [00:00<?, ?it/s]

In [1]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoProcessor, AutoTokenizer, AutoModelForCausalLM

class VisionLanguageModel(nn.Module):
    def __init__(self, vision_encoder_ckpt, language_model_ckpt, tokenizer, modality_input_dim=768, modality_output_dim=576):
        super().__init__()
        self.vision_encoder = AutoModel.from_pretrained(vision_encoder_ckpt).vision_model
        self.vision_processor = AutoProcessor.from_pretrained(vision_encoder_ckpt)
        self.modality_projector = nn.Linear(modality_input_dim, modality_output_dim, bias=False)
        self.tokenizer = tokenizer # AutoTokenizer.from_pretrained(language_model_ckpt, extra_special_tokens={"image_token": "<|image|>"})
        self.llm = AutoModelForCausalLM.from_pretrained(language_model_ckpt)
        self.llm.resize_token_embeddings(len(self.tokenizer))  # Resize the LLM's token embeddings to match the tokenizer's new vocab size

    def _replace_img_tokens_with_embd(self, input_ids, token_embd, image_embd):
        """
        Replace every image-token placeholder in `input_ids` with the corresponding slice
        from `image_embd`. Supports an arbitrary number of image-token placeholders per sample.
        The first example in the batch might have 2 images and the second none.
        """
        # Clone the original embeddings to avoid in-place issues
        updated_token_embd = token_embd.clone()

        # Build a mask of all image-token positions: shape [B, T_seq]
        mask = (input_ids == self.tokenizer.image_token_id)
        updated_token_embd[mask] = image_embd.view(-1, image_embd.size(-1)).to(updated_token_embd.dtype) # torch flattens before assigning

        return updated_token_embd

    def forward(self, input_ids, image):
        processed_img = self.vision_processor(images=[image], return_tensors="pt").to(self.llm.device)
        image_embd = self.vision_encoder(**processed_img).last_hidden_state
        image_embd = self.modality_projector(image_embd)

        token_embd = self.llm.model.embed_tokens(input_ids)
        combined_embd = self._replace_img_tokens_with_embd(input_ids, token_embd, image_embd)

        logits = self.llm(inputs_embeds=combined_embd).logits

        # 1. Create labels from the input_ids
        labels = input_ids.clone()

        # 2. We don't want to compute loss on image tokens.
        #    Set them to the ignore_index (which is pad_token_id).
        labels[labels == self.tokenizer.image_token_id] = self.tokenizer.pad_token_id

        # 3. Standard next-token-prediction shifting:
        #    The logit at index `i` predicts the token at index `i+1`
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        # 4. Compute loss, ignoring all tokens marked with pad_token_id
        #    (this now includes both padding AND image tokens)
        loss = nn.functional.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1),
            ignore_index=self.tokenizer.pad_token_id
        )

        return logits, loss

In [19]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch.optim as optim
import time
import json
import numpy as np

vision_encoder_ckpt = "google/siglip2-base-patch16-256"
language_model_ckpt = "HuggingFaceTB/SmolLM2-135M-Instruct"
dataset = load_dataset("HuggingFaceM4/FineVisionMax", split="train", streaming=True)
tokenizer = AutoTokenizer.from_pretrained(language_model_ckpt, extra_special_tokens={"image_token": "<|image|>"})
vlm = VisionLanguageModel(vision_encoder_ckpt, language_model_ckpt, tokenizer).to("cuda")
optimizer = optim.AdamW(vlm.parameters(), lr=1e-4)

def tokenize_sample(sample):
    messages_list = []
    for data_dict in sample['texts']:
        messages_list.append({'role': 'user', 'content': data_dict['user']})
        messages_list.append({'role': 'assistant', 'content': data_dict['assistant']})
    messages_list[0]['content'] = tokenizer.image_token*256 + messages_list[0]['content']  # add image tokens
    tokens = tokenizer.apply_chat_template(messages_list, tokenize=True)
    return tokens['input_ids']

step = 0
losses = []
max_length = 2048
curr_sample = []
curr_image = []
batch = []
image_batch = []
max_batch_size = 4
batch_complete = False
start = time.time()
for sample in dataset:
    if len(sample['images']) != 1:
      continue
    tokens = tokenize_sample(sample)
    if len(tokens) > max_length:
        continue  # Skip samples that are too long
    if len(curr_sample) + len(tokens) < max_length:
        curr_sample.extend(tokens)
        curr_image.extend([sample['images'][0].convert('RGB')])
    else:
        padded = torch.nn.functional.pad(
            torch.tensor(curr_sample, dtype=torch.long),
            (0, max_length - len(curr_sample)), # <--- Right padding
            "constant",
            tokenizer.pad_token_id,
        )
        batch.append(padded)
        image_batch.extend(curr_image)
        if len(batch) == max_batch_size:
          batch_complete = True # We've reached the max batch size, so we can continue
        curr_sample = tokens
        curr_image = [sample['images'][0].convert('RGB')]
    if batch_complete:
      step += 1
      if step%100 == 0:
        print(f"step {step} | loss {loss.item():.4f} | smoothed loss: {np.mean(losses[-10:]):.4f}")
        print(f"The last 100 steps took: {time.time()-start:.2f} seconds.")
        start = time.time()

      optimizer.zero_grad()
      logits, loss = vlm(torch.stack(batch).long().cuda(), image_batch)
      losses.append(loss.item())

      loss.backward()
      optimizer.step()

      batch_complete = False
      batch = []
      image_batch = []
      if step > 5000:
        break

json.dump([round(l, 4) for l in losses], open("chapter_3_batched_loss.json", "w"))

Resolving data files:   0%|          | 0/10000 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (28674 > 8192). Running this sequence through the model will result in indexing errors


step 100 | loss 1.4531 | smoothed loss: 1.5258
The last 100 steps took: 91.58 seconds.
step 200 | loss 1.4609 | smoothed loss: 1.4172
The last 100 steps took: 93.71 seconds.
step 300 | loss 1.5703 | smoothed loss: 1.3359
The last 100 steps took: 93.15 seconds.
step 400 | loss 1.5156 | smoothed loss: 1.3262
The last 100 steps took: 93.01 seconds.
step 500 | loss 1.0859 | smoothed loss: 1.2203
The last 100 steps took: 73.74 seconds.
step 600 | loss 1.3281 | smoothed loss: 1.3719
The last 100 steps took: 96.48 seconds.
step 700 | loss 0.6992 | smoothed loss: 1.2375
The last 100 steps took: 92.60 seconds.
step 800 | loss 1.2656 | smoothed loss: 1.3758
The last 100 steps took: 95.97 seconds.
step 900 | loss 1.1484 | smoothed loss: 1.2699
The last 100 steps took: 84.72 seconds.
step 1000 | loss 1.5000 | smoothed loss: 1.3703
The last 100 steps took: 75.39 seconds.
step 1100 | loss 0.8594 | smoothed loss: 1.0289
The last 100 steps took: 85.08 seconds.
step 1200 | loss 1.4141 | smoothed loss: 

In [4]:
from transformers.image_utils import load_image

image = load_image("https://huggingface.co/datasets/merve/coco/resolve/main/val2017/000000002006.jpg")

def generate(trained_vlm, tokenizer, input_text, image, max_new_tokens):
    text = [{"role": "user", "content": tokenizer.image_token*256 + input_text}]
    text_str = tokenizer.apply_chat_template(text, add_generation_prompt=True, tokenize=False)
    input_tokens = tokenizer(text_str, return_tensors="pt").input_ids.cuda()    
    next_token = None
    output = []
    end_of_sentence_token = tokenizer.eos_token_id

    while len(output) < max_new_tokens:
        prediction = trained_vlm(input_tokens, image)[0]
        next_token = torch.argmax(prediction[:, -1, :])
        if next_token == end_of_sentence_token:
            break
        output.append(next_token)
        input_tokens = torch.cat([input_tokens, next_token[None, None]], dim=1)

    return ''.join(tokenizer.batch_decode(output))

print(generate(vlm, tokenizer, "What do you see here?", image, 32))


In the image, I can see a person standing in the middle of the scene. The person is wearing a black and white striped shirt with a red stripe running


In [8]:
import torch
import time
import numpy as np
from PIL import Image

def create_fake_batch(tokenizer, batch_size=4, seq_len=2048):
    input_ids = torch.randint(
        low=0,
        high=len(tokenizer),
        size=(batch_size, seq_len),
        dtype=torch.long,
    )

    # insert image tokens at the beginning
    input_ids[:, :256] = tokenizer.image_token_id

    # create dummy images
    image = Image.new("RGB", (256, 256), color="white")
    image_batch = [image for _ in range(batch_size)]

    return input_ids, image_batch


def benchmark_model(vlm, tokenizer, device="cuda",
                    steps=50, warmup=10):

    vlm = vlm.to(device)
    vlm.train()

    input_ids, image_batch = create_fake_batch(tokenizer)
    input_ids = input_ids.to(device)

    optimizer = torch.optim.AdamW(vlm.parameters(), lr=1e-4)

    # ----------------------
    # warmup
    # ----------------------
    for _ in range(warmup):
        logits, loss = vlm(input_ids, image_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    torch.cuda.synchronize()

    # ----------------------
    # measure inference
    # ----------------------
    vlm.eval()
    times = []

    with torch.no_grad():
        for _ in range(steps):
            start = time.time()

            logits, loss = vlm(input_ids, image_batch)

            torch.cuda.synchronize()
            times.append(time.time() - start)

    inference_latency = np.mean(times)
    tokens = input_ids.numel()
    inference_tps = tokens / inference_latency

    # ----------------------
    # measure training
    # ----------------------
    vlm.train()
    times = []

    for _ in range(steps):
        start = time.time()

        logits, loss = vlm(input_ids, image_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        torch.cuda.synchronize()
        times.append(time.time() - start)

    train_latency = np.mean(times)
    train_tps = tokens / train_latency

    return {
        "inference_latency_sec": inference_latency,
        "inference_tokens_per_sec": inference_tps,
        "train_latency_sec": train_latency,
        "train_tokens_per_sec": train_tps,
    }

benchmark_model(vlm, tokenizer)

{'inference_latency_sec': np.float64(0.16778892040252685),
 'inference_tokens_per_sec': np.float64(48823.247568119106),
 'train_latency_sec': np.float64(0.5832825183868409),
 'train_tokens_per_sec': np.float64(14044.652019841531)}